In [ ]:
import os

import h5py
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

full_dataframe = pd.DataFrame()
full_itof = pd.DataFrame()
for f in sorted(os.listdir(r"J:\ctgroup\Edward\DATA\VMI\20251112\1,5 s 1,4 m PO med scan")):
    if not f.endswith('.cv4'):
        continue
    pos = int(f.split(".")[0])
    print(pos)
    file = os.path.join(r"J:\ctgroup\Edward\DATA\VMI\20251112\1,5 s 1,4 m PO med scan", f)
    data = {}
    with h5py.File(file, 'r') as f:
        for key in f.keys():
            data[key] = np.array(f[key])

        clusters = pd.DataFrame({
            'x': data['x'],
            'y': data['y'],
            'toa': data['t'],
            'pulse_id': data['cluster_corr'],
        })

        etof = pd.DataFrame({
            'etof': data['t_etof'],
            'pulse_id': data['etof_corr'],
        })

        itof = pd.DataFrame({
            'itof': data['t_tof'],
            'pulse_id': data['tof_corr'],
        })
    del data, f, key

    combined = clusters.merge(etof, on='pulse_id')
    combined = combined[combined['toa'] > 0]
    del combined['pulse_id']
    combined['position'] = pos
    full_dataframe = pd.concat([full_dataframe, combined], ignore_index=True)
    itof.drop(columns=['pulse_id'], inplace=True)
    full_itof = pd.concat([full_itof, itof], ignore_index=True)

In [ ]:
px.histogram(full_dataframe, x='etof', nbins=500, title='Electron Time-of-Flight Histogram').show()

px.histogram(full_dataframe, x='toa', nbins=500, title='ToA Histogram').show()



In [ ]:
roi_etof = (400, 600)
roi_toa = (200, 400)

roi_data = full_dataframe[
    (full_dataframe['etof'] >= roi_etof[0]) & (full_dataframe['etof'] <= roi_etof[1]) &
    (full_dataframe['toa'] >= roi_toa[0]) & (full_dataframe['toa'] <= roi_toa[1])
    ]
fig = make_subplots(rows=1, cols=2, subplot_titles=('Electron Time-of-Flight Histogram (ROI)', 'ToA Histogram (ROI)'))
fig.add_trace(
        go.Histogram(x=roi_data['etof'], nbinsx=500, name='ETOF (ROI)'),
        row=1, col=1
)
fig.add_trace(
        go.Histogram(x=roi_data['toa'], nbinsx=500, name='ToA (ROI)'),
        row=1, col=2
)

fig.show()

px.density_heatmap(
        roi_data, x='etof', y='toa', nbinsx=500, nbinsy=500, color_continuous_scale=px.colors.sequential.Inferno,
        title='2D Histogram of ETOF vs ToA (ROI)',
        width=800, height=800,
).show()

In [ ]:
roi_data['diff'] = roi_data['etof'] - roi_data['toa']
px.histogram(
        roi_data, x='diff', nbins=500,
        title='Histogram of ETOF - ToA (ROI)',
        log_y=True,
        width=800, height=600,
).show()

hist, bin_edges = np.histogram(roi_data['diff'], bins=500, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2


def gaussian(x, amp, mean, stddev):
    return amp * np.exp(-((x - mean) ** 2) / (2 * stddev ** 2))


from scipy.optimize import curve_fit

popt, pcov = curve_fit(gaussian, bin_centers, hist, p0=[max(hist), np.mean(roi_data['diff']), np.std(roi_data['diff'])])
fit_x = np.linspace(min(bin_centers), max(bin_centers), 1000)
fit_y = gaussian(fit_x, *popt)

px.histogram(
        roi_data, x='diff', nbins=500,
        title='Histogram of ETOF - ToA (ROI) with Gaussian Fit',
        width=800, height=600, histnorm='probability',
).add_scatter(
        x=fit_x, y=fit_y, mode='lines', name='Gaussian Fit', line=dict(color='red')
).show()
print(f"Fitted Gaussian parameters: Amplitude={popt[0]}, Mean={popt[1]}, Stddev={popt[2]}")

In [ ]:
px.density_heatmap(
        roi_data[roi_data['diff'].between(220, 250) & roi_data['etof'].between(485, 515)],
        x='etof', y='diff',
        nbinsx=100, nbinsy=100,
        color_continuous_scale=px.colors.sequential.Inferno,
        title='2D Histogram of ETOF vs (ETOF - ToA) (ROI)',
        width=800, height=800,
)

In [ ]:
px.histogram(
        full_itof[full_itof['itof'].between(0, 16000)], x='itof', nbins=500,
)